In [1]:
import os
import logging
import warnings
import subprocess
import xagg
import glob as glob
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import netCDF4 as nc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from ecmwfapi import ECMWFService
from dateutil.relativedelta import relativedelta
import cfgrib
import dask

In [7]:
def retrieve_ifs_forecast(target_dir, start, end, params, init_times, lead_times, bounds):
    '''
    Downloads IFS forecast data from ECMWF MARS archive
    
    Inputs:
        target_dir: parent directory to save data within (str)
        start: start date (str, YYYY-MM-DD)
        end: end date (str, YYYY-MM-DD)
        params: dictionary of parameter names and codes {str: str}
        init_times: list of initalization times (list of str)
        lead_times: list of lead times (list of str)
        bounds: latitude and longitude boundaries [deg N, deg W, deg S, deg E] (list of str)
    Outputs:
        None
    '''
    # Create subdirectories
    dates = pd.date_range(start=start, end=end, freq='MS')
    for param in params.keys():
        for init_time in init_times:
            for lead_time in lead_times:
                for date in dates:
                    year = date.strftime("%Y")
                    month = date.strftime("%m")
                    path = os.path.join(target_dir, param, init_time, lead_time, year, month)
                    os.makedirs(path, exist_ok=True)
    
    # Establish API connection to ECMWF MARS archive
    server = ECMWFService("mars")
    
    # Retrieve data
    dates = pd.date_range(start=start, end=end, freq='D')
    for param in params.keys():
        for init_time in init_times:
            for lead_time in lead_times:
                for date in dates:
                    date_str = date.strftime("%Y-%m-%d")
                    valid_time = date + relativedelta(hours=int(init_time)) + relativedelta(hours=int(lead_time))
                    valid_year = valid_time.strftime("%Y")
                    valid_month = valid_time.strftime("%m")
                    valid_str = valid_time.strftime("%Y_%m_%d_%H")
                    path = os.path.join(target_dir, param, init_time, lead_time, valid_year, valid_month, f'ifs_fc_{param}_{valid_str}z.grib')
                    if os.path.exists(path[:-4]+"nc"): # Skip already downloaded data
                        logging.info(f'Skipping already downloaded data {path[:-4]+"nc"}')
                        continue
                    try:
                        server.execute({
                            'class': "od",
                            'type': "fc",
                            'stream': "oper",
                            'expver': "1",
                            'repres': "gg",
                            'levtype': "sfc",
                            'param': params[param],
                            'time': init_time,
                            'step': lead_time,
                            'domain': "g",
                            'resol': "auto",
                            'area': "/".join(bounds),
                            'grid': "F1280",
                            'padding': "0",
                            'expect': "any",
                            'date': date_str
                        },
                            path)
                        
                    except Exception as e:
                        logging.error(f'Unable to retrieve forecast data for {"_".join([param,init_time,lead_time,date.strftime("%Y-%m-%d")])}: {e}')
                        continue
    
                    # Convert to netcdf using eccodes (module load ecccodes)
                    subprocess.run(f'grib_to_netcdf -o {path[:-4]+"nc"} {path}', shell=True)
                
                    # Delete grib file
                    subprocess.run(['rm', path])
    return

In [8]:
retrieve_ifs_forecast(
    target_dir = '/Users/dcalhoun/Desktop/Research/Data/ecmwf/ifs',
    start = '2016-01-01',
    end = '2016-01-01',
    params = {'t2m': '167.128'},
    init_times = ['00', '12'],
    lead_times = ['24'],
    bounds = ['45', '-85', '35', '-70']
)

2025-03-09 23:53:30 ECMWF API python library 1.6.3
2025-03-09 23:53:30 ECMWF API at https://api.ecmwf.int/v1
2025-03-09 23:53:31 Welcome Dean Calhoun
2025-03-09 23:53:32 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2025-03-09 23:53:33 Request submitted
2025-03-09 23:53:33 Request id: 67ce623da8bddec2c3f8cb53
2025-03-09 23:53:33 Request is submitted
2025-03-09 23:53:34 Request is active
2025-03-09 23:53:45 Calling 'nice mars /tmp/20250310-0350/16/tmp-_mars-gcJFAE.req'
2025-03-09 23:53:45 Forcing MIR_CACHE_PATH=/data/ec_coeff
2025-03-09 23:53:45 mars - WARN -
2025-03-09 23:53:45 mars - WARN -
2025-03-09 23:53:45 MIR environment variables:
2025-03-09 23:53:45 MIR_CACHE_PATH=/data/ec_coeff
2025-03-09 23:53:45 MIR_LSM_NAMED=1km.climate.v013
2025-03-09 23:53:45 Using MARS binary: /usr/local/apps/mars/versions/6.33.19.4/bin/mars.bin
2025-03-09 23:53:45 mars - INFO   - 20250310.035334 - Welcome to MARS
2025-03-09 23:

In [33]:
def aggregate_to_geography(ds, gdf, outfile, bounds=None):
    '''
    Aggregates gridded data to a specified geography.
    
    Inputs:
    	ds: gridded data (xarray dataset)
        gdf: geography to aggregate to (geopandas geodataframe)
        outfile: path to save aggregated data (str)
        bounds: latitude and longitude boundaries [deg N, deg W, deg S, deg E] (list of int)
    Outputs:
    	None
    '''
    logging.info('Starting aggregation')
    if bounds is not None:
        gdf = gdf.cx[bounds[1]:bounds[3], bounds[0]:bounds[2]]
    weightmap = xagg.pixel_overlaps(ds,gdf)
    aggregated = xagg.aggregate(ds,weightmap)
    with warnings.catch_warnings(action="ignore"):
        try:
            aggregated.to_shp(outfile)
            logging.info('Completed aggregation')
        except Exception:
            print('Failed to save to .shp')
    return aggregated

In [34]:
shapefile_path = '/Users/dcalhoun/Desktop/Research/Data/census/shapefiles/nhgis0001_shapefile_tl2023_us_county_2023/US_county_2023.shp'
outfile = '/Users/dcalhoun/Desktop/Research/Data/aggregated/test_t2m.shp'
bounds = ['45', '-85', '35', '-70']

## Read shapefile
gdf = gpd.read_file(shapefile_path).to_crs('WGS84')
gdf["INTPTLAT"] = pd.to_numeric(gdf["INTPTLAT"])
gdf["INTPTLON"] = pd.to_numeric(gdf["INTPTLON"])

## Read data and aggregate
path = '/Users/dcalhoun/Desktop/Research/Data/ecmwf/ifs/t2m/*/24/2016/01/*.nc'
ds = xr.open_mfdataset(glob.glob(path))
aggregated = aggregate_to_geography(ds=ds, gdf=gdf, outfile=outfile, bounds=bounds)

creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
aggregating t2m...
all variables aggregated to polygons!
Failed to save to .shp


In [35]:
# aggregated = gpd.read_file(outfile)
# aggregated.plot(column='t2m', legend=True, cmap='coolwarm', figsize=(10,10))

In [43]:
aggregated.agg

,GISJOIN,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,...,INTPTLAT,INTPTLON,Shape_Leng,Shape_Area,ORIG_FID,poly_idx,rel_area,pix_idxs,coords,t2m
0,G0901100,09,110,02830244,09110,0500000US09110,Capitol,Capitol Planning Region,PL,H5,...,41.816970,-72.575886,279784.074636,2.709609e+09,2559,0,"[[3.083661657511248e-05, 0.011491040015426357,...","[19768, 19980, 19981, 19982, 19983, 19986, 199...","[(41.51142120361328, -72.84364318847656), (41....","[[[273.7805940718943, 270.807881279154]]]"
1,G0901200,09,120,02830245,09120,0500000US09120,Greater Bridgeport,Greater Bridgeport Planning Region,PL,H5,...,41.184558,-73.209401,142636.005460,3.715984e+08,2560,1,"[[0.0024724772405494986, 0.024664216033242214,...","[18484, 18696, 18697, 18698, 18699, 18909, 189...","[(41.089630126953125, -73.26554107666016), (41...","[[[275.80461148025194, 272.3213241078694]]]"
2,G0901300,09,130,02830246,09130,0500000US09130,Lower Connecticut River Valley,Lower Connecticut River Valley Planning Region,PL,H5,...,41.420307,-72.493525,217319.886623,1.146362e+09,2561,2,"[[0.00035648368233256837, 0.000939198293051085...","[18920, 18921, 18922, 18923, 19132, 19133, 191...","[(41.230228424072266, -72.5623779296875), (41....","[[[275.79429900731236, 271.6865593347528]]]"
3,G0901400,09,140,02830249,09140,0500000US09140,Naugatuck Valley,Naugatuck Valley Planning Region,PL,H5,...,41.518931,-73.087906,192741.207897,1.091854e+09,2562,3,"[[0.0023115238242755235, 0.0001858397595559657...","[18912, 18913, 19124, 19125, 19126, 19337, 193...","[(41.230228424072266, -73.12490844726562), (41...","[[[274.11804373884064, 270.93773281150493]]]"
4,G0901500,09,150,02830250,09150,0500000US09150,Northeastern Connecticut,Northeastern Connecticut Planning Region,PL,H5,...,41.842375,-71.972655,181914.015560,1.458569e+09,2563,4,"[[0.009158488234681671, 0.007885024311408785, ...","[19782, 19783, 19995, 19996, 20204, 20205, 202...","[(41.51142120361328, -71.85921478271484), (41....","[[[273.58226483392724, 270.8920132003286]]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
768,G5401010,54,101,01550057,54101,0500000US54101,Webster,Webster County,06,H1,...,38.483459,-80.449051,168171.457666,1.440626e+09,2463,768,"[[0.001758949943070824, 0.004326313469262776, ...","[9650, 9861, 9862, 9863, 10072, 10073, 10074, ...","[(38.20738220214844, -80.3674545288086), (38.2...","[[[270.9314211393701, 269.0091156943078]]]"
769,G5401030,54,103,01717836,54103,0500000US54103,Wetzel,Wetzel County,06,H1,...,39.598180,-80.635399,169549.627843,9.358391e+08,2247,769,"[[0.0009535384934477052, 0.0013710367700046618...","[13268, 13269, 13479, 13480, 13481, 13482, 134...","[(39.40245819091797, -80.57840728759766), (39....","[[[272.68705922113384, 270.3102991210885]]]"
770,G5401050,54,105,01678877,54105,0500000US54105,Wirt,Wirt County,06,H1,...,39.020034,-81.382975,133633.308469,6.080408e+08,2677,770,"[[0.0002968175703894346, 0.021044456738700076,...","[11763, 11764, 11765, 11766, 11767, 11976, 119...","[(38.910369873046875, -81.56282806396484), (38...","[[[273.61758224582405, 270.99209513833694]]]"
771,G5401070,54,107,01560558,54107,0500000US54107,Wood,Wood County,06,H1,...,39.211602,-81.516234,149969.708708,9.758206e+08,2461,771,"[[0.006774045062475124, 0.026738996329293177, ...","[12187, 12188, 12189, 12190, 12399, 12400, 124...","[(39.05096435546875, -81.70346069335938), (39....","[[[273.8363030257004, 271.06584040570567]]]"


In [38]:
ds

<xarray.Dataset> Size: 485kB
Dimensions:    (time: 2, latitude: 142, longitude: 213)
Coordinates:
  * longitude  (longitude) float32 852B -84.94 -84.87 -84.8 ... -70.1 -70.03
  * latitude   (latitude) float32 568B 44.96 44.89 44.82 ... 35.18 35.11 35.04
  * time       (time) datetime64[ns] 16B 2016-01-02 2016-01-02T12:00:00
Data variables:
    t2m        (time, latitude, longitude) float64 484kB dask.array<chunksize=(1, 142, 213), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2025-03-10 03:53:46 GMT by grib_to_netcdf-2.39.0: grib_to_n...